# bench

> Self-retrieval eval harness: sample indexed chunks, use their docstring (or name stems) as the query, and check whether search finds the source chunk. Absolute scores are optimistic — the docstring text lives inside the indexed chunk — so treat every number as **relative**: compare rerank on/off, `fast` vs `accurate` profiles, or before/after a chunking change. This is the kosha equivalent of semble's published NDCG@10 benchmark loop.

In [ ]:
#| default_exp bench

In [ ]:
#| export
import time, random, math, re
from json import loads as jl
from fastcore.all import L, Path
from kosha.core import Kosha
from kosha.rank import split_ident

## Sampling queries

In [ ]:
#| export
def sample_queries(k:Kosha,
	store:str = 'code',      # which store to sample from: code (repo) or env (packages)
	n:int = 100,             # number of eval queries
	seed:int = 42,           # shuffle seed, keep fixed for A/B comparisons
	min_len:int = 20,        # min query length in chars
	mode:str = 'docstring'   # docstring = first docstring line as query; name = identifier stems as query
) -> L:
	'Sample (query, id, mod_name) eval triples from an indexed store.'
	st = k.code_st if store == 'code' else k.env_st
	rows = st(select='id,metadata', where="json_extract(metadata,'$.docstring') IS NOT NULL")
	out, seen = [], set()
	for r in rows:
		m = jl(r['metadata']) if isinstance(r['metadata'], str) else r['metadata']
		doc = (m.get('docstring') or '').strip()
		q = doc.splitlines()[0] if mode == 'docstring' and doc else ' '.join(split_ident(m.get('name') or ''))
		q = re.sub(r'[^\w\s]', ' ', q).strip()  # strip FTS5 query syntax (col:term, quotes, slashes)
		if len(q) < min_len or q in seen: continue
		seen.add(q)
		out.append(dict(q=q, id=r['id'], mod_name=m.get('mod_name')))
	random.Random(seed).shuffle(out)
	return L(out[:n])

## Metrics

> Single-relevant-document versions of the standard metrics: NDCG@10 reduces to `1/log2(rank+2)`, MRR to `1/(rank+1)`.

In [ ]:
#| export
def _rank_of(res, tid, mod=None):
	'Rank (0-based) of the target chunk in a result list, matched by id then mod_name.'
	for i, r in enumerate(res):
		if r.get('id') == tid: return i
		m = r.get('metadata') or {}
		if mod and isinstance(m, dict) and m.get('mod_name') == mod: return i
	return None

def run_bench(k:Kosha,
	queries,             # eval triples from sample_queries
	limit:int = 10,      # retrieval depth
	rerank:bool = True,  # code-aware rerank on/off
	repo:bool = True,    # search the repo store
	env:bool = True      # search the env store
) -> dict:
	"""Self-retrieval eval: NDCG@10, Recall@1/5/10, MRR, mean query latency. Absolute numbers are
	inflated (the docstring text is part of the indexed chunk) — use for A/B comparisons only:
	rerank on/off, embedding profiles, chunking changes."""
	rec, ndcg, mrr, lat = {1:0, 5:0, 10:0}, 0.0, 0.0, []
	for qq in queries:
		t0 = time.perf_counter()
		res = k.context(qq['q'], limit=max(limit, 10), repo=repo, env=env, graph=False, rerank=rerank,
		                columns='id,content,metadata', auto_sync=False)
		lat.append((time.perf_counter() - t0)*1000)
		rk = _rank_of(res, qq['id'], qq.get('mod_name'))
		if rk is None: continue
		for c in rec:
			if rk < c: rec[c] += 1
		if rk < 10: ndcg += 1.0/math.log2(rk + 2)
		mrr += 1.0/(rk + 1)
	n = len(queries) or 1
	return dict(n=len(queries), ndcg10=round(ndcg/n, 4), recall1=round(rec[1]/n, 4),
	            recall5=round(rec[5]/n, 4), recall10=round(rec[10]/n, 4), mrr=round(mrr/n, 4),
	            ms_per_query=round(sum(lat)/max(len(lat), 1), 1))

## A/B comparisons

In [ ]:
#| export
def compare_rerank(k:Kosha, n:int = 100, store:str = 'code') -> dict:
	'A/B the code-aware rerank against plain RRF on the same index.'
	qs = sample_queries(k, store=store, n=n)
	return dict(rerank=run_bench(k, qs), no_rerank=run_bench(k, qs, rerank=False))

def compare_profiles(dir=None,
	pkgs:list = None,               # packages to index per profile; None = repo only
	n:int = 100,                    # eval queries
	profiles=('fast','accurate')    # embedding profiles to compare
) -> dict:
	"""A/B embedding profiles. Builds a separate throwaway index per profile (embeds everything
	once per profile) — expensive; run on a small repo, and leave pkgs=None to skip packages."""
	import tempfile
	out = {}
	for pf in profiles:
		td = Path(tempfile.mkdtemp(prefix=f'kosha-bench-{pf}-'))
		k = Kosha(dir=dir, xdg_dir=td/'xdg', db_dir=td/'code', profile=pf)
		t0 = time.perf_counter()
		k.update_repo(verbose=False)
		if pkgs: k.update_pkgs(pkgs, verbose=False)
		idx_s = round(time.perf_counter() - t0, 2)
		qs = sample_queries(k, n=n)
		out[pf] = run_bench(k, qs) | dict(index_seconds=idx_s)
	return out